# 05 — Riparian Buffer Encroachment Analysis

**Goal:** the second half of the project's stated objective — quantify built-up encroachment
into riparian zones along Nairobi's rivers. Uses the single-year 2024 Random Forest built-up
classification (notebook 03, ~86% held-out accuracy) rather than the 2019→2024 change map:
notebook 04 found that cross-date comparison unreliable even after masking and normalization, so
this notebook deliberately stays with the one classification layer that's actually trustworthy —
a snapshot of *where* built-up land currently sits relative to rivers, not *when* it arrived.

**River geometry:** `WWF/HydroSHEDS/v1/FreeFlowingRivers`, filtered to reaches intersecting
Nairobi — 42 reaches, 393 km total length. Checked before committing to this dataset: the
nearest reach to Nairobi's CBD is only 688 m away with a discharge (0.22 m³/s) consistent with a
small urban stream, not a major river skirting the city's edge — so this dataset does capture the
real urban river network (Nairobi River, Ngong River, and tributaries), not just larger regional
rivers passing through the outskirts.

**Buffer width:** swept at 30/50/100 m rather than fixed at one number, matching notebook 02's
threshold-sweep approach. These are analysis buffers for this notebook, not a legal
determination of Kenya's statutory riparian reserve (which is tiered by river size under the
Water Act / NEMA guidelines) — that distinction is worth being explicit about since this is a
"riparian encroachment" project.

In [1]:
import sys
sys.path.insert(0, '../src')

import ee
import geemap
import urllib.request
from acquisition import get_nairobi_boundary, get_sentinel2_composite
from classification import (
    build_feature_image, get_worldcover_builtup, sample_training_points,
    train_random_forest, classify_builtup,
)

ee.Initialize(project='solar-haven-349708')

nairobi = get_nairobi_boundary()
rivers = ee.FeatureCollection('WWF/HydroSHEDS/v1/FreeFlowingRivers').filterBounds(nairobi)
dist_to_river = rivers.distance(searchRadius=200, maxError=10).clip(nairobi)

print('River reaches in Nairobi:', rivers.size().getInfo())
print('Total reach length (km):', rivers.aggregate_sum('LENGTH_KM').getInfo())

River reaches in Nairobi: 42


Total reach length (km): 393.42400000000015


## Classify 2024 built-up (same procedure as notebook 03)

In [2]:
composite, scene_count = get_sentinel2_composite(
    nairobi, start_date='2024-06-01', end_date='2024-09-30', cloud_threshold=20
)
features = build_feature_image(composite)
worldcover_builtup = get_worldcover_builtup(nairobi)
train_samples, test_samples = sample_training_points(features, worldcover_builtup, nairobi)
classifier = train_random_forest(train_samples)
builtup = classify_builtup(features, classifier)

test_accuracy = test_samples.classify(classifier).errorMatrix(
    'builtup', 'classification'
).accuracy().getInfo()
print(f'Scenes: {scene_count}, held-out accuracy (sanity check): {test_accuracy * 100:.1f}%')

Scenes: 11, held-out accuracy (sanity check): 85.9%


## City-wide comparison: riparian buffer vs. rest of the city

If rivers attract built-up encroachment, buffer zones should show a *higher* built-up fraction
than the city average. Nairobi National Park is excluded from this comparison via a raster mask
(not a vector `geometry.difference()`, which is expensive in Earth Engine and hung when tried) —
several river reaches run along or through the park, which is entirely non-built-up by law and
would mechanically pull any riparian statistic down regardless of real encroachment elsewhere.

In [3]:
wdpa = ee.FeatureCollection('WCMC/WDPA/current/polygons')
park_fc = wdpa.filter(ee.Filter.And(ee.Filter.eq('NAME', 'Nairobi'), ee.Filter.gt('REP_AREA', 100)))
outside_park_mask = ee.Image().paint(park_fc, 1).unmask(0).eq(0)

builtup_outside_park = builtup.updateMask(outside_park_mask)

city_frac = builtup_outside_park.rename('b').reduceRegion(
    reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9, bestEffort=True
).getInfo()['b'] * 100
print(f'City-wide built-up fraction (excl. park): {city_frac:.1f}%')

print(f'{"Buffer (m)":>10} {"In-buffer %":>12} {"Outside %":>10}')
for buf in [30, 50, 100]:
    zone = dist_to_river.lte(buf)
    grouped = builtup_outside_park.rename('b').addBands(zone.rename('zone')).reduceRegion(
        reducer=ee.Reducer.mean().group(groupField=1, groupName='zone'),
        geometry=nairobi, scale=10, maxPixels=1e9, bestEffort=True
    ).getInfo()['groups']
    by_zone = {g['zone']: g['mean'] * 100 for g in grouped}
    print(f'{buf:>10} {by_zone.get(1, float("nan")):>11.1f}% {by_zone.get(0, float("nan")):>9.1f}%')

City-wide built-up fraction (excl. park): 43.8%
Buffer (m)  In-buffer %  Outside %


        30        41.0%      41.8%


        50        40.9%      41.9%


       100        41.0%      42.3%


## Diagnosing a surprising result

The city-wide sweep shows riparian buffer zones with a built-up fraction *close to or slightly
below* the rest of the city — not the elevated encroachment signal the "informal settlements
crowd riverbanks" narrative predicts. Rather than report that flat aggregate as "no
encroachment," check it against specific, well-documented informal-settlement river corridors:
Mathare and Kibera are two of Nairobi's largest informal settlements, both built directly along
river valleys (Mathare River and Ngong River respectively). If the city-wide average is masking
a real, spatially concentrated effect, these should show it.

In [4]:
hotspots = {
    'Mathare': (36.857, -1.259),
    'Kibera': (36.789, -1.313),
    'Mukuru': (36.870, -1.310),
}
zone_30 = dist_to_river.lte(30)

print(f'{"Location":>10} {"Riverside (30m) %":>18} {"Surrounding 1km %":>18} {"Diff":>8}')
for name, (lon, lat) in hotspots.items():
    region = ee.Geometry.Point([lon, lat]).buffer(1000)
    grouped = builtup.rename('b').addBands(zone_30.rename('zone')).reduceRegion(
        reducer=ee.Reducer.mean().group(groupField=1, groupName='zone'),
        geometry=region, scale=10, maxPixels=1e9, bestEffort=True
    ).getInfo()['groups']
    by_zone = {g['zone']: g['mean'] * 100 for g in grouped}
    riverside, surrounding = by_zone.get(1, float('nan')), by_zone.get(0, float('nan'))
    print(f'{name:>10} {riverside:>17.1f}% {surrounding:>17.1f}% {riverside - surrounding:>+7.1f}pp')

  Location  Riverside (30m) %  Surrounding 1km %     Diff


   Mathare              89.6%              80.0%    +9.6pp


    Kibera              96.6%              88.1%    +8.5pp


    Mukuru              97.7%              98.4%    -0.7pp


## Visualize

In [5]:
builtup_vis = {'min': 0, 'max': 1, 'palette': ['black', 'red']}
river_vis = {'min': 0, 'max': 100, 'palette': ['00FFFF', '000000']}

Map = geemap.Map(center=[-1.290, 36.868], zoom=11)
Map.addLayer(composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'True color', False)
Map.addLayer(builtup, builtup_vis, '2024 built-up')
Map.addLayer(dist_to_river.lte(30).selfMask(), {'palette': ['cyan']}, 'Riparian buffer (30m)')
Map.addLayer(rivers, {'color': 'blue'}, 'River reaches (HydroSHEDS)')
for name, (lon, lat) in hotspots.items():
    Map.addLayer(ee.Geometry.Point([lon, lat]), {'color': 'yellow'}, name)
Map.addLayerControl()
Map

Map(center=[-1.29, 36.868], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [6]:
composite_riparian = builtup.visualize(**builtup_vis).blend(
    dist_to_river.lte(30).selfMask().visualize(palette=['00FFFF'], opacity=0.6)
)
url = composite_riparian.getThumbURL({'region': nairobi, 'dimensions': 900})
path = '../data/processed/nairobi_riparian_buffer_builtup.png'
urllib.request.urlretrieve(url, path)
print(f'Saved {path}')

Saved ../data/processed/nairobi_riparian_buffer_builtup.png


## Summary

| Buffer | In-buffer built-up % | Rest-of-city % |
|---|---|---|
| 30 m | 41.0% | 41.8% |
| 50 m | 40.9% | 41.9% |
| 100 m | 41.0% | 42.3% |

**City-wide (excl. Nairobi National Park):** riparian buffer zones do *not* show elevated
built-up fraction relative to the rest of Nairobi — actually marginally lower (~41.0% vs.
41.8-42.3%), contrary to the naive expectation that rivers uniformly attract encroachment.

| Location | Riverside (30m) | Surrounding 1km | Diff |
|---|---|---|---|
| Mathare | 89.6% | 80.0% | **+9.6pp** |
| Kibera | 96.6% | 88.1% | **+8.5pp** |
| Mukuru | 97.7% | 98.4% | -0.7pp |

**Hotspot-level:** Mathare and Kibera — two of Nairobi's largest, most-documented informal
settlements, both built along river valleys — show a clear positive riverside effect: built-up
fraction within 30m of the river runs 8-10 percentage points higher than in the surrounding 1km.
Mukuru shows no differential, because the surrounding area is already near-saturated built-up
(97-98%) regardless of distance to the river — there's no "outside" to be less built-up than.

**Conclusion: riparian encroachment in Nairobi is a localized phenomenon concentrated in specific
informal settlements along river corridors, not a uniform city-wide pattern.** The city-wide
aggregate alone would have missed this entirely (and would have reported the *opposite* of the
real story if taken at face value) — the finding only surfaced by checking known hotspots
specifically. That's a real methodological lesson for a monitoring tool meant to be useful:
a single city-wide dashboard number can hide the exact hotspots the tool is supposed to catch.

**Caveats:**
- This is a single-year snapshot (2024), not change detection — it identifies *where* built-up
  land currently sits close to rivers, not *when* it arrived or whether encroachment is actively
  worsening (notebook 04 found year-over-year comparison unreliable with the current pipeline).
- HydroSHEDS' reach network, while validated as reasonably complete for Nairobi's main urban
  rivers, may still miss the smallest tributaries and drainage channels informal settlements
  sometimes back directly onto.
- Buffer widths (30/50/100m) are analytical choices for this notebook, not Kenya's legal riparian
  reserve determination, which is tiered by river size under the Water Act / NEMA guidelines.
- The three hotspots were chosen because they're well-documented in the literature as riverside
  informal settlements, not discovered algorithmically — a systematic city-wide scan (e.g.
  gridded riverside-vs-surrounding comparison, flagging cells above some threshold) would be the
  natural next step to find hotspots without relying on prior knowledge of where to look.